In [1]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [2]:
evaluation_questions = [
    {
        "question": "What is retrieval-augmented generation?",
        "expected_category": "RAG"
    },
    {
        "question": "How does watermark-based incremental loading work in Azure Data Factory?",
        "expected_category": "Azure Data Factory"
    },
    {
        "question": "What is the difference between narrow and wide transformations in Spark?",
        "expected_category": "Spark"
    },
    {
        "question": "What is the purpose of a fact table in a star schema?",
        "expected_category": "Data Modeling"
    },
    {
        "question": "Why is idempotency important in incremental data pipelines?",
        "expected_category": "Data Engineering"
    },
    {
    "question": "What are the main features of Kubernetes?",
    "expected_category": None
    }
]

print(f"Evaluation questions prepared: {len(evaluation_questions)}")

Evaluation questions prepared: 6


In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv(override=True)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

EMBEDDING_MODEL = "text-embedding-3-large"
GENERATION_MODEL = "gpt-5.4-mini"
TOP_K = 3

chroma_client = chromadb.PersistentClient(path="vector_store")

collection = chroma_client.get_or_create_collection(
    name="technical_docs"
)

print("Evaluation setup ready.")
print("Stored records:", collection.count())

Evaluation setup ready.
Stored records: 37


In [4]:
def evaluate_question(question, expected_category=None):
    # Create question embedding
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=[question]
    )

    question_embedding = response.data[0].embedding

    # Retrieve top-k chunks
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=TOP_K
    )

    # Build context
    context = "\n\n".join(results["documents"][0])

    prompt = f"""
Answer the question using only the retrieved context below.

If the retrieved context does not contain enough information,
say that the available documents do not provide enough information.

Question:
{question}

Retrieved Context:
{context}
"""

    # Generate grounded answer
    generation = client.responses.create(
        model=GENERATION_MODEL,
        input=prompt
    )

    answer = generation.output_text

    retrieved_categories = [
        metadata["category"]
        for metadata in results["metadatas"][0]
    ]

    sources = [
        metadata["source_file"]
        for metadata in results["metadatas"][0]
    ]

    return {
        "question": question,
        "expected_category": expected_category,
        "retrieved_categories": retrieved_categories,
        "sources": sources,
        "answer": answer
    }

print("Evaluation function ready.")

Evaluation function ready.


In [5]:
evaluation_results = []

for item in evaluation_questions:
    result = evaluate_question(
        question=item["question"],
        expected_category=item["expected_category"]
    )

    evaluation_results.append(result)

print(f"Evaluation completed for {len(evaluation_results)} questions.")

Evaluation completed for 6 questions.


In [6]:
for i, result in enumerate(evaluation_results, start=1):
    print(f"\nQuestion {i}: {result['question']}")
    print("Expected Category:", result["expected_category"])
    print("Retrieved Categories:", result["retrieved_categories"])
    print("Sources:", result["sources"])
    print("-" * 70)


Question 1: What is retrieval-augmented generation?
Expected Category: RAG
Retrieved Categories: ['RAG', 'RAG', 'RAG']
Sources: ['rag_fundamentals.pdf', 'advanced_rag.pdf', 'rag_fundamentals.pdf']
----------------------------------------------------------------------

Question 2: How does watermark-based incremental loading work in Azure Data Factory?
Expected Category: Azure Data Factory
Retrieved Categories: ['Data Engineering', 'Azure Data Factory', 'Azure Data Factory']
Sources: ['incremental_data_pipelines.pdf', 'azure_data_factory.pdf', 'azure_data_factory.pdf']
----------------------------------------------------------------------

Question 3: What is the difference between narrow and wide transformations in Spark?
Expected Category: Spark
Retrieved Categories: ['Spark', 'Spark', 'Azure Databricks']
Sources: ['pyspark_and_spark.pdf', 'pyspark_and_spark.pdf', 'azure_databricks.pdf']
----------------------------------------------------------------------

Question 4: What is the p

In [7]:
correct = 0
evaluated = 0

for result in evaluation_results:
    expected = result["expected_category"]

    if expected is None:
        continue

    retrieved = result["retrieved_categories"]
    hit = expected in retrieved

    evaluated += 1

    if hit:
        correct += 1

    print(
        result["question"],
        "→",
        "PASS" if hit else "FAIL"
    )

hit_rate = correct / evaluated

print(f"\nRetrieval Hit@3: {hit_rate:.0%}")
print(f"Questions evaluated for Hit@3: {evaluated}")

What is retrieval-augmented generation? → PASS
How does watermark-based incremental loading work in Azure Data Factory? → PASS
What is the difference between narrow and wide transformations in Spark? → PASS
What is the purpose of a fact table in a star schema? → PASS
Why is idempotency important in incremental data pipelines? → PASS

Retrieval Hit@3: 100%
Questions evaluated for Hit@3: 5


In [8]:
top1_pass = 0
top1_questions = 0

for result in evaluation_results:
    expected_category = result["expected_category"]

    if expected_category is None:
        continue

    top1_questions += 1

    top1_category = result["retrieved_categories"][0]
    correct_top1 = top1_category == expected_category

    if correct_top1:
        top1_pass += 1

    print(
        result["question"],
        "→",
        "PASS" if correct_top1 else "FAIL"
    )

top1_accuracy = top1_pass / top1_questions

print(f"\nTop-1 Category Accuracy: {top1_accuracy:.0%}")
print(f"Questions evaluated for Top-1: {top1_questions}")

What is retrieval-augmented generation? → PASS
How does watermark-based incremental loading work in Azure Data Factory? → FAIL
What is the difference between narrow and wide transformations in Spark? → PASS
What is the purpose of a fact table in a star schema? → PASS
Why is idempotency important in incremental data pipelines? → PASS

Top-1 Category Accuracy: 80%
Questions evaluated for Top-1: 5


In [9]:
out_of_scope_result = next(
    result
    for result in evaluation_results
    if result["expected_category"] is None
)

answer_text = out_of_scope_result["answer"].lower()

grounding_pass = (
    "do not provide enough information" in answer_text
    or "does not provide enough information" in answer_text
    or "not enough information" in answer_text
)

print("Out-of-Scope Question:")
print(out_of_scope_result["question"])

print("\nGenerated Answer:")
print(out_of_scope_result["answer"])

print(
    "\nGrounding Check:",
    "PASS" if grounding_pass else "FAIL"
)

Out-of-Scope Question:
What are the main features of Kubernetes?

Generated Answer:
The available documents do not provide enough information about Kubernetes.

Grounding Check: PASS
